# Smart Face Recognition Attendance System - Integrated Frontend

This notebook provides an integrated UI for the backend attendance system.
- Directly imports backend modules
- Real-time status updates
- Error handling and logging

In [3]:
# Install required packages
import subprocess
import sys

def install_packages():
    packages = [
        'ipywidgets',
        'opencv-python-headless',
        'face-recognition',
        'pandas',
        'openpyxl',
        'pillow',
        'imutils',
        'dlib',
        'numpy'
    ]
    for package in packages:
        try:
            __import__(package.replace('-', '_'))
        except ImportError:
            print(f"Installing {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_packages()
print("✓ All packages installed successfully")

Installing opencv-python-headless...
Installing pillow...
✓ All packages installed successfully


In [4]:
# Import required libraries
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import sys
import os
from datetime import datetime

# Add project to path
sys.path.insert(0, '..')

# Import backend modules
from register import StudentRegistration
from attendance import AttendanceMarker
from trainer import train_from_dataset
from utils import load_encodings, ensure_directories

print("✓ Backend modules imported successfully")
ensure_directories()
print("✓ Directories ensured")

✓ Backend modules imported successfully
✓ Directories ensured


In [5]:
# Global variables
current_section = 'menu'
output_area = widgets.Output()
status_area = widgets.Output()

# Create main UI components
title = widgets.HTML(
    value="<h1 style='text-align: center; color: #2c3e50;'>🎓 Smart Face Recognition Attendance System</h1>"
)

subtitle = widgets.HTML(
    value="<h3 style='text-align: center; color: #34495e;'>With Anti-Spoofing (Blink Detection)</h3><hr>"
)

# Status/Info display
def update_system_info():
    encodings_data = load_encodings()
    total_students = len(encodings_data['names'])
    return f"<p><strong>Registered Students:</strong> {total_students}</p>"

info_label = widgets.HTML(value=update_system_info())

print("✓ UI components created")

✓ UI components created


In [6]:
# Registration function
def register_student(name, roll_number, student_class):
    """Complete registration process"""
    try:
        with status_area:
            clear_output()
            print("🔄 Validating input...")
            
            register = StudentRegistration()
            valid, message = register.validate_input(name, roll_number, student_class)
            
            if not valid:
                print(f"❌ Validation Failed: {message}")
                return
            
            print(f"✓ Validation passed")
            print(f"📝 Registering: {name} (Roll: {roll_number})")
            print(f"📸 Capturing faces...\nPlease look at the camera and stay still.")
            print(f"Press ESC in the camera window to stop.\n")
            
            # Capture faces
            success, message = register.capture_faces(name, roll_number, student_class)
            
            if not success:
                print(f"❌ Capture Failed: {message}")
                return
            
            print(f"✓ {message}")
            print(f"🔧 Generating face encodings...")
            
            # Generate encodings
            success, message = register.generate_encodings(name, roll_number, student_class)
            
            if success:
                print(f"✅ Registration successful!")
                print(f"   Name: {name}")
                print(f"   Roll: {roll_number}")
                print(f"   Class: {student_class}")
                print(f"   {message}")
            else:
                print(f"❌ Encoding Failed: {message}")
    
    except Exception as e:
        with status_area:
            clear_output()
            print(f"❌ Error: {str(e)}")
            import traceback
            traceback.print_exc()

# Attendance function
def start_attendance():
    """Start attendance marking"""
    try:
        with status_area:
            clear_output()
            print("🔍 Starting attendance marking...")
            print("👀 Look at the camera and blink naturally")
            print("🔐 Anti-spoofing enabled (Blink detection)")
            print("Press ESC in the camera window to stop.\n")
            
            attendance = AttendanceMarker()
            attendance.run_attendance()
            
            print(f"✅ Attendance session completed")
            print(f"📊 Total marked: {len(attendance.marked_today)}")
    
    except Exception as e:
        with status_area:
            clear_output()
            print(f"❌ Error: {str(e)}")
            import traceback
            traceback.print_exc()

# Training function
def train_system():
    """Train face encodings from dataset"""
    try:
        with status_area:
            clear_output()
            print("🔄 Training face encodings from dataset...")
            
            train_from_dataset()
            
            encodings_data = load_encodings()
            print(f"✅ Training complete!")
            print(f"📊 Total students: {len(encodings_data['names'])}")
            print(f"💾 Total encodings: {len(encodings_data['encodings'])}")
            
            # Update info
            info_label.value = update_system_info()
    
    except Exception as e:
        with status_area:
            clear_output()
            print(f"❌ Error: {str(e)}")
            import traceback
            traceback.print_exc()

print("✓ Backend functions wrapped")

✓ Backend functions wrapped


In [7]:
# Create UI widgets and layouts

# Main buttons
register_btn = widgets.Button(
    description='📝 Register Student',
    button_style='info',
    layout=widgets.Layout(width='250px', height='50px', margin='10px')
)

attendance_btn = widgets.Button(
    description='📸 Take Attendance',
    button_style='success',
    layout=widgets.Layout(width='250px', height='50px', margin='10px')
)

train_btn = widgets.Button(
    description='🔄 Train Encodings',
    button_style='warning',
    layout=widgets.Layout(width='250px', height='50px', margin='10px')
)

back_btn = widgets.Button(
    description='🔙 Back to Menu',
    button_style='primary',
    layout=widgets.Layout(width='250px', height='40px', margin='10px')
)

# Registration form
name_input = widgets.Text(
    placeholder='Enter student name',
    description='Name:',
    layout=widgets.Layout(width='400px')
)

roll_input = widgets.Text(
    placeholder='Enter roll number',
    description='Roll No:',
    layout=widgets.Layout(width='400px')
)

class_input = widgets.Text(
    placeholder='Enter class',
    description='Class:',
    layout=widgets.Layout(width='400px')
)

register_submit = widgets.Button(
    description='✅ Start Registration',
    button_style='success',
    layout=widgets.Layout(width='250px', height='50px')
)

print("✓ UI widgets created")

✓ UI widgets created


In [2]:
# Button click handlers

def show_menu():
    """Show main menu"""
    global current_section
    current_section = 'menu'
    
    with output_area:
        clear_output()
        display(widgets.VBox([
            widgets.HTML("<h3>📋 Main Menu</h3>"),
            register_btn,
            attendance_btn,
            train_btn,
            info_label
        ]))

def show_registration_form():
    """Show registration form"""
    global current_section
    current_section = 'registration'
    
    with output_area:
        clear_output()
        display(widgets.VBox([
            widgets.HTML("<h3>📝 Student Registration</h3>"),
            name_input,
            roll_input,
            class_input,
            widgets.HTML("<p style='color: #7f8c8d;'>After submission, camera will open for face capture.</p>"),
            register_submit,
            back_btn
        ]))

def on_register_click(b):
    show_registration_form()

def on_attend_click(b):
    with output_area:
        clear_output()
        display(widgets.VBox([
            widgets.HTML("<h3>📸 Attendance Marking</h3>"),
            widgets.HTML("<p>Starting attendance process...</p>"),
            back_btn
        ]))
    start_attendance()
    info_label.value = update_system_info()

def on_train_click(b):
    with output_area:
        clear_output()
        display(widgets.VBox([
            widgets.HTML("<h3>🔄 Training Encodings</h3>"),
            widgets.HTML("<p>Training in progress...</p>"),
            back_btn
        ]))
    train_system()
    info_label.value = update_system_info()

def on_back_click(b):
    show_menu()

def on_submit_registration(b):
    register_student(name_input.value, roll_input.value, class_input.value)
    # Clear inputs
    name_input.value = ''
    roll_input.value = ''
    class_input.value = ''
    info_label.value = update_system_info()

# Attach handlers
register_btn.on_click(on_register_click)
attendance_btn.on_click(on_attend_click)
train_btn.on_click(on_train_click)
back_btn.on_click(on_back_click)
register_submit.on_click(on_submit_registration)

print("✓ Event handlers attached")

NameError: name 'register_btn' is not defined

In [14]:
# Create system information box
info_box = widgets.HTML(
    value="""
    <div style='background-color: #ecf0f1; padding: 15px; border-radius: 5px; margin-top: 20px;'>
        <h4 style='color: #2c3e50;'>ℹ️ System Information:</h4>
        <ul>
            <li>📁 Dataset folder: <code>dataset/</code></li>
            <li>📊 Attendance file: <code>attendance/attendance.xlsx</code></li>
            <li>🔍 Liveness detection: Blink-based</li>
            <li>👥 Multiple face support: Yes</li>
            <li>⚙️ Backend: Direct module integration</li>
        </ul>
    </div>
    """
)

print("✓ Info box created")

✓ Info box created


In [1]:
# Main display
main_layout = widgets.VBox([
    title,
    subtitle,
    output_area,
    status_area,
    info_box
])

# Display the UI
display(main_layout)

# Show initial menu
show_menu()

print("\n✅ Smart Face Recognition Attendance System - Ready!")

NameError: name 'widgets' is not defined